# 제주 특산물 가격 예측 - CPU 최적 트리 앙상블 v4.0.0

| 항목 | 내용 |
|------|------|
| **버전** | v4.0.0 |
| **날짜** | 2026-03-13 |
| **모델** | CatBoost + LightGBM 앙상블 (XGBoost 제거) |
| **검증** | 2022년 3월 (테스트와 동일 계절) |
| **전처리** | v1.0.1 + 사이클 피처(sin/cos) + 주말/분기/연일 |
| **CPU** | i7-1360P 16스레드 × 80% = 12스레드 |
| **출력** | results/submission_v4.0.0.csv |

## v4.0.0 변경 내용 (v3.1.0 대비)
| 항목 | v3.1.0 | v4.0.0 |
|------|--------|--------|
| DNN | Embedding + Residual | **제거** (CPU 느림, 일반화 취약) |
| XGBoost | 포함 | **제거** (내부 MAE 687, 가장 낮은 성능) |
| LightGBM | 없음 | **신규 추가** (CPU 최고 속도) |
| 앙상블 | scipy 최적화 가중치 | **단순 평균** (과적합 방지) |
| 검증 전략 | 80/20 시간 분할 | **2022년 3월** (테스트 계절 일치) |
| 피처 | 기본 시간 피처 | **사이클 인코딩·주말·분기** 추가 |
| CPU 설정 | n_jobs=-1 | **n_jobs=12** (80% 사용) |

---
## 버전 히스토리 & DACON 제출 결과 분석

### 버전별 성능 요약

| 버전 | 모델 | 핵심 변경 | 내부 검증 MAE | DACON Public | DACON Private |
|------|------|----------|--------------:|-------------:|--------------:|
| **v1.0.0** | DNN | 기본 DNN, 전처리 미흡 | — | 1695.6 | 1721.4 |
| **v1.0.1** | DNN | DACON 1위 전처리(공휴일·누적 week_num·TG sqrt·후처리) + 기본 Dense | 522 원/kg | **658.6** ✅ | **825.0** ✅ |
| **v1.1.0** | DNN+Emb | item·corporation·location을 Embedding 레이어로 처리 | 537 원/kg | 미제출 | 미제출 |
| **v1.2.0** | DNN+Emb+Res | Embedding + Residual Block(skip connection) 추가 | 532 원/kg | 1155.3 ❌ | 1326.3 ❌ |
| **v2.0.0** | LSTM | 14일 슬라이딩 윈도우 + 오토레그레시브 예측 | 890 원/kg | 783.9 | 1030.2 |
| **v3.0.0** | DNN+CB+XGB | DNN·CatBoost·XGBoost 3모델 앙상블 도입 | ~510 원/kg | 미제출 | 미제출 |
| **v3.1.0** | DNN(Res)+CB+XGB | DNN을 Residual 구조(v1.2.0)로 교체한 앙상블 | ~510 원/kg | 910.9 ❌ | 1062.6 ❌ |
| **v4.0.0** | **LGB+CB+XGB** | DNN 제거, LightGBM 추가 → CPU 최적 순수 트리 앙상블 | 510 원/kg | — | — |

---

### 문제 분석

#### 1. 단순 모델(v1.0.1)이 복잡한 모델보다 강한 이유 — 과적합
- 내부 검증 MAE: v1.0.1(522) ≈ v1.2.0(532) ≈ v3.1.0(~510) → 비슷해 보이지만
- DACON Public: v1.0.1(658) << v1.2.0(1155), v3.1.0(910) → 실제로는 v1.0.1이 압도적
- **복잡한 모델(Embedding, Residual, Ensemble)이 검증 데이터에 과적합**되어 일반화 실패
- v1.2.0은 public(1155)보다 private(1326)이 더 나쁨 → 테스트 전체 기간에서도 일반화 실패

#### 2. 검증 전략 문제 — 시간 분포 불일치
- 현재 80/20 시간순 분할 → 검증 세트가 2022년 후반~2023년 초
- **실제 테스트는 2023년 3월 한 달**로 특정 계절·시장 패턴을 가짐
- 검증 MAE가 낮아도 실제 성능을 보장하지 못함
- **개선안**: 2023년 3월과 동일한 기간(전년도 3월)을 검증셋으로 사용

#### 3. 앙상블 가중치 과적합 (v3.1.0)
- scipy로 최적화한 가중치가 검증 세트 분포에 과도하게 맞춰짐
- 검증에서 MAE ~510이지만 Public 910.9 → **가중치 자체가 검증셋에 오버핏**
- v1.0.1(단일 모델)보다 앙상블이 오히려 나쁜 역설적 결과
- **개선안**: 균등 가중치(1/3, 1/3, 1/3) 또는 단순 평균 앙상블 시도

#### 4. 트리 모델의 가능성
- v4.0.0 내부 검증: CatBoost 단독 MAE 512 (v1.0.1 DNN 522보다 낮음)
- **CatBoost 단독이 v1.0.1보다 좋을 수 있음** → 앙상블보다 CatBoost 단독 제출도 고려

---

### v4.0.0 방향
> 위 분석을 바탕으로 트리 모델만으로 앙상블 구성, 가중치는 scipy 최적화 유지.  
> 만약 Public 점수가 v1.0.1보다 나쁘면 **CatBoost 단독 제출**을 우선 시도할 것.

---
## 1. 라이브러리 로드

In [1]:
try:
    import lightgbm as lgb
    print(f'LightGBM: {lgb.__version__}')
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', 'lightgbm'], check=True)
    import lightgbm as lgb
    print(f'LightGBM installed: {lgb.__version__}')

LightGBM installed: 4.6.0


In [18]:
#pip list

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings, datetime, random, os, platform
warnings.filterwarnings('ignore')

import holidays
import lightgbm as lgb
from catboost import CatBoostRegressor

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:
    plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

SEED = 2024
random.seed(SEED); np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

# CPU 80% 사용 (i7-1360P: 16스레드 → 12스레드)
N_CPU_TOTAL = os.cpu_count()
N_JOBS = max(1, int(N_CPU_TOTAL * 0.8))
print(f'총 CPU 스레드: {N_CPU_TOTAL}  →  사용 스레드: {N_JOBS} (80%)')
print(f'LightGBM: {lgb.__version__} | CatBoost: {__import__("catboost").__version__}')

---
## 2. 데이터 로드

In [3]:
DATA_PATH = '../data/'
train = pd.read_csv(DATA_PATH + 'train.csv', encoding='utf-8-sig')
test  = pd.read_csv(DATA_PATH + 'test.csv',  encoding='utf-8-sig')
sub   = pd.read_csv(DATA_PATH + 'sample_submission.csv', encoding='utf-8-sig')
print(f'train: {train.shape}, test: {test.shape}')

train: (59397, 7), test: (1092, 5)


---
## 3. 전처리 + 이상치 + 타겟 변환 (v1.0.1 동일)

In [ ]:
def add_features(df):
    """사이클 인코딩 + 추가 시간 피처"""
    df['quarter']      = df['timestamp'].dt.quarter
    df['is_weekend']   = (df['week_day'] >= 5).astype(int)
    df['day_of_year']  = df['timestamp'].dt.dayofyear

    df['month_sin']       = np.sin(2 * np.pi * df['month']       / 12)
    df['month_cos']       = np.cos(2 * np.pi * df['month']       / 12)
    df['week_day_sin']    = np.sin(2 * np.pi * df['week_day']    / 7)
    df['week_day_cos']    = np.cos(2 * np.pi * df['week_day']    / 7)
    df['day_of_year_sin'] = np.sin(2 * np.pi * df['day_of_year'] / 365)
    df['day_of_year_cos'] = np.cos(2 * np.pi * df['day_of_year'] / 365)
    return df

def pre_all(train, test):
    train['timestamp'] = pd.to_datetime(train['timestamp'])
    test['timestamp']  = pd.to_datetime(test['timestamp'])
    df = pd.concat([train, test]).reset_index(drop=True)
    df.rename(columns={'supply(kg)': 'supply', 'price(원/kg)': 'price'}, inplace=True)

    df['year']     = df['timestamp'].dt.year
    df['month']    = df['timestamp'].dt.month
    df['day']      = df['timestamp'].dt.day
    df['week_day'] = df['timestamp'].dt.weekday

    le = LabelEncoder()
    df['year_month'] = df['timestamp'].map(lambda x: f'{x.year}-{x.month}')
    df['year_month'] = le.fit_transform(df['year_month'])

    df['week'] = df['timestamp'].map(
        lambda x: datetime.datetime(x.year, x.month, x.day).isocalendar()[1]
    )
    week_offsets = {2019: 0, 2020: 52, 2021: 52+53, 2022: 52+53+53, 2023: 52+53+53+52}
    df['week_num'] = df.apply(lambda r: int(r['week']) + week_offsets.get(r['year'], 0), axis=1)
    df.loc[df['timestamp'] == '2019-12-30', 'week_num'] = 52
    df.loc[df['timestamp'] == '2019-12-31', 'week_num'] = 52

    kr_holi = holidays.KR()
    df['holiday'] = df['timestamp'].map(lambda x: 1 if x in kr_holi else 0)

    df = add_features(df)

    train_out = df[~df['price'].isnull()].sort_values('timestamp').reset_index(drop=True)
    test_out  = df[ df['price'].isnull()].sort_values('timestamp').reset_index(drop=True)
    print(f'전처리 완료 — train: {train_out.shape}, test: {test_out.shape}')
    return train_out, test_out

train_pre, test_pre = pre_all(train, test)

# 이상치 처리
outlier_thresholds = {'TG': 20000, 'RD': 5000, 'BC': 8000, 'CB': 2300}
for item, thr in outlier_thresholds.items():
    idx = train_pre[(train_pre['item'] == item) & (train_pre['price'] > thr)].index
    if len(idx):
        mean_p = train_pre[(train_pre['item'] == item) & (train_pre['price'] != 0)]['price'].mean()
        train_pre.loc[idx, 'price'] = mean_p
        print(f'{item}: {len(idx)}개 이상치 → 평균({mean_p:.0f})')

# 타겟 변환
train_pre['price_transformed'] = np.where(
    train_pre['item'] == 'TG',
    np.sqrt(train_pre['price']),
    np.log1p(train_pre['price'])
)

# TG 공휴일 보정
tg_mask = (train_pre['item'] == 'TG') & (train_pre['holiday'] == 1) & (train_pre['price'] != 0)
active_holi = list(train_pre[tg_mask].groupby('timestamp').count().reset_index()['timestamp'])
fix_idx = train_pre[train_pre['timestamp'].isin(active_holi)].index
train_pre.loc[fix_idx, 'holiday'] = 0
print(f'TG 공휴일 보정: {len(fix_idx)}개')
print(f'추가 피처: quarter, is_weekend, day_of_year, sin/cos(month, week_day, day_of_year)')

---
## 4. 검증 분리 (2022년 3월) + 유틸 함수

> 테스트 데이터는 2023년 3월 → 동일 계절인 **2022년 3월**을 검증셋으로 사용  
> 기존 80/20 시간 분할보다 테스트 분포를 잘 반영하여 검증 MAE 신뢰도 향상

In [ ]:
TARGET_TRF = 'price_transformed'
TARGET_COL = 'price'

CAT_COLS  = ['item', 'corporation', 'location']
TIME_COLS = ['year_month', 'week_num', 'week_day', 'month', 'day', 'holiday', 'year',
             'quarter', 'is_weekend', 'day_of_year',
             'month_sin', 'month_cos', 'week_day_sin', 'week_day_cos',
             'day_of_year_sin', 'day_of_year_cos']
TREE_FEAT = TIME_COLS + CAT_COLS

# 2022년 3월을 검증셋으로 분리 (테스트와 동일한 계절)
val_mask = (train_pre['year'] == 2022) & (train_pre['month'] == 3)
train_df = train_pre[~val_mask].reset_index(drop=True)
val_df   = train_pre[ val_mask].reset_index(drop=True)

y_tr      = train_df[TARGET_TRF].values
y_vl      = val_df[TARGET_TRF].values
y_vl_orig = val_df[TARGET_COL].values
is_tg_vl  = (val_df['item'] == 'TG').values

def inverse_transform(y_trf, is_tg):
    result = np.where(
        is_tg.astype(bool),
        np.power(np.clip(y_trf, 0, None), 2),
        np.expm1(y_trf)
    )
    return np.clip(result, 0, None)

def eval_model(y_true, y_pred, name):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    print(f'  [{name}]  MAE={mae:>8,.1f}  RMSE={rmse:>8,.1f}  R²={r2:.4f}')
    return mae, rmse, r2

print(f'학습셋: {len(train_df):,}행  |  검증셋: {len(val_df):,}행 (2022-03)')
print(f'검증 기간: {val_df["timestamp"].min().date()} ~ {val_df["timestamp"].max().date()}')

---
## 5. 모델 1 — LightGBM

> CPU에서 가장 빠른 트리 모델. leaf-wise 성장 전략, n_jobs=N_JOBS (80% 활용)

In [ ]:
X_lgb_tr   = train_df[TREE_FEAT].copy()
X_lgb_vl   = val_df[TREE_FEAT].copy()
X_lgb_test = test_pre[TREE_FEAT].copy()

# LightGBM 카테고리 피처: pandas category dtype으로 변환
for col in CAT_COLS:
    cats = pd.Categorical(
        pd.concat([X_lgb_tr[col], X_lgb_vl[col], X_lgb_test[col]])
    ).categories
    X_lgb_tr[col]   = pd.Categorical(X_lgb_tr[col],   categories=cats)
    X_lgb_vl[col]   = pd.Categorical(X_lgb_vl[col],   categories=cats)
    X_lgb_test[col] = pd.Categorical(X_lgb_test[col], categories=cats)

lgb_model = lgb.LGBMRegressor(
    n_estimators=5000,
    learning_rate=0.03,
    num_leaves=31,
    max_depth=7,
    min_child_samples=50,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=2.0,
    objective='regression_l1',
    random_state=SEED,
    n_jobs=N_JOBS,
    verbose=-1
)

lgb_model.fit(
    X_lgb_tr, y_tr,
    eval_set=[(X_lgb_vl, y_vl)],
    callbacks=[
        lgb.early_stopping(stopping_rounds=200, verbose=False),
        lgb.log_evaluation(period=500)
    ]
)
print(f'LightGBM 최적 반복: {lgb_model.best_iteration_}')

---
## 6. 모델 2 — CatBoost

> 카테고리 피처 내장 처리, thread_count=N_JOBS (80% 활용)

In [ ]:
X_cb_tr   = train_df[TREE_FEAT]
X_cb_vl   = val_df[TREE_FEAT]
X_cb_test = test_pre[TREE_FEAT]

cb_model = CatBoostRegressor(
    iterations=5000,
    learning_rate=0.03,
    depth=7,
    l2_leaf_reg=3,
    loss_function='MAE',
    eval_metric='MAE',
    cat_features=CAT_COLS,
    random_seed=SEED,
    early_stopping_rounds=200,
    thread_count=N_JOBS,
    verbose=500
)
cb_model.fit(X_cb_tr, y_tr, eval_set=(X_cb_vl, y_vl), use_best_model=True)
print(f'CatBoost 최적 반복: {cb_model.get_best_iteration()}')

---
## 7. 개별 모델 검증 성능

In [ ]:
# LightGBM 예측
lgb_vl_pred_trf = lgb_model.predict(X_lgb_vl)
lgb_vl_pred     = inverse_transform(lgb_vl_pred_trf, is_tg_vl)

# CatBoost 예측
cb_vl_pred_trf  = cb_model.predict(X_cb_vl)
cb_vl_pred      = inverse_transform(cb_vl_pred_trf, is_tg_vl)

print('=' * 60)
print('  개별 모델 검증 성능 (2022년 3월)')
print('=' * 60)
lgb_scores = eval_model(y_vl_orig, lgb_vl_pred, 'LightGBM')
cb_scores  = eval_model(y_vl_orig, cb_vl_pred,  'CatBoost')
print('=' * 60)

---
## 8. 앙상블 (단순 평균) + 최적 모델 자동 선택

> scipy 가중치 최적화는 검증셋 과적합 위험이 있으므로 **단순 평균** 사용  
> CB 단독 / LGB 단독 / 평균 앙상블 중 검증 MAE 최소 모델을 자동 선택

In [ ]:
# 단순 평균 앙상블
ens_vl_pred = (lgb_vl_pred + cb_vl_pred) / 2

mae_lgb = mean_absolute_error(y_vl_orig, lgb_vl_pred)
mae_cb  = mean_absolute_error(y_vl_orig, cb_vl_pred)
mae_ens = mean_absolute_error(y_vl_orig, ens_vl_pred)

print(f'  LightGBM 단독  MAE: {mae_lgb:>8,.2f}')
print(f'  CatBoost 단독  MAE: {mae_cb:>8,.2f}')
print(f'  단순 평균 앙상블 MAE: {mae_ens:>8,.2f}')

# 검증 MAE 최소 모델 자동 선택
best_mae  = min(mae_lgb, mae_cb, mae_ens)
if best_mae == mae_ens:
    FINAL_MODE = 'ensemble'
    final_vl_pred = ens_vl_pred
elif best_mae == mae_cb:
    FINAL_MODE = 'catboost'
    final_vl_pred = cb_vl_pred
else:
    FINAL_MODE = 'lightgbm'
    final_vl_pred = lgb_vl_pred

print(f'\n→ 최종 선택: {FINAL_MODE.upper()}  (MAE={best_mae:,.2f})')

In [ ]:
mae  = mean_absolute_error(y_vl_orig, final_vl_pred)
rmse = np.sqrt(mean_squared_error(y_vl_orig, final_vl_pred))
r2   = r2_score(y_vl_orig, final_vl_pred)

print('=' * 60)
print(f'  최종 모델({FINAL_MODE.upper()}) 검증 성능 (2022년 3월)')
print('=' * 60)
print(f'  MAE  : {mae:>10,.2f} 원/kg')
print(f'  RMSE : {rmse:>10,.2f} 원/kg')
print(f'  R²   : {r2:>10.4f}')
print('=' * 60)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

model_names = ['LightGBM', 'CatBoost', 'Ensemble(avg)']
maes   = [mae_lgb, mae_cb, mae_ens]
colors = ['steelblue', 'darkorange', 'crimson']
best_idx = maes.index(min(maes))

bars = axes[0].bar(model_names, maes, color=colors, alpha=0.85, edgecolor='white')
bars[best_idx].set_edgecolor('gold')
bars[best_idx].set_linewidth(3)
axes[0].set_title('모델별 검증 MAE (2022년 3월)', fontsize=13)
axes[0].set_ylabel('MAE (원/kg)')
axes[0].grid(axis='y', alpha=0.3)
for bar, val in zip(bars, maes):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                 f'{val:,.0f}', ha='center', va='bottom', fontsize=10)

max_v = max(y_vl_orig.max(), final_vl_pred.max())
axes[1].scatter(y_vl_orig, final_vl_pred, alpha=0.3, s=8, color=colors[best_idx])
axes[1].plot([0, max_v], [0, max_v], 'k--', lw=2)
axes[1].set_title(f'{FINAL_MODE.upper()} 실제 vs 예측  R²={r2:.4f}', fontsize=12)
axes[1].set_xlabel('실제'); axes[1].set_ylabel('예측'); axes[1].grid(alpha=0.3)

plt.suptitle('v4.0.0 검증 결과 (2022-03)', fontsize=15, fontweight='bold')
plt.tight_layout()
os.makedirs('./results', exist_ok=True)
plt.savefig('./results/ensemble_eval_v4.0.0.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 9. 피처 중요도

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# LightGBM
lgb_imp = pd.Series(lgb_model.feature_importances_, index=TREE_FEAT).sort_values(ascending=False)
lgb_imp.head(12).plot(kind='barh', ax=axes[0], color='steelblue', alpha=0.85)
axes[0].invert_yaxis()
axes[0].set_title('LightGBM Top-12 Feature Importance', fontsize=12)
axes[0].grid(axis='x', alpha=0.3)

# CatBoost
cb_imp_vals = cb_model.get_feature_importance()
cb_imp = pd.Series(cb_imp_vals, index=TREE_FEAT).sort_values(ascending=False)
cb_imp.head(12).plot(kind='barh', ax=axes[1], color='darkorange', alpha=0.85)
axes[1].invert_yaxis()
axes[1].set_title('CatBoost Top-12 Feature Importance', fontsize=12)
axes[1].grid(axis='x', alpha=0.3)

plt.suptitle('모델별 피처 중요도', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('./results/feature_importance_v4.0.0.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 10. 테스트 예측 + 후처리 + 제출

In [ ]:
lgb_test_trf = lgb_model.predict(X_lgb_test)
cb_test_trf  = cb_model.predict(X_cb_test)

is_tg_test = (test_pre['item'] == 'TG').values
lgb_test_pred = inverse_transform(lgb_test_trf, is_tg_test)
cb_test_pred  = inverse_transform(cb_test_trf,  is_tg_test)

# 선택된 모드에 따라 최종 예측 생성
if FINAL_MODE == 'ensemble':
    final_test_pred = (lgb_test_pred + cb_test_pred) / 2
elif FINAL_MODE == 'catboost':
    final_test_pred = cb_test_pred
else:
    final_test_pred = lgb_test_pred

final_test_pred = np.clip(final_test_pred, 0, None)

result_df = test_pre[['ID', 'item']].copy()
result_df['answer'] = final_test_pred

min_thresholds = {'TG': 400, 'CB': 50, 'RD': 10, 'CR': 150, 'BC': 100}
for item, thr in min_thresholds.items():
    mask = (result_df['item'] == item) & (result_df['answer'] < thr)
    result_df.loc[mask, 'answer'] = 0.0
    if mask.sum() > 0:
        print(f'{item}: {mask.sum()}개 → 0 처리')

print('\n예측 통계:')
print(result_df.groupby('item')['answer'].agg(['mean','min','max']).round(1))

In [ ]:
result = sub[['ID']].merge(result_df[['ID', 'answer']], on='ID', how='left')
result['answer'] = result['answer'].fillna(0.0)

SUBMISSION_PATH = './results/submission_v4.0.0.csv'
result.to_csv(SUBMISSION_PATH, index=False, encoding='utf-8-sig')
print(f'저장: {SUBMISSION_PATH}, 행: {len(result)}')
result.head(10)

---
## 11. 결과 요약

In [ ]:
print('=' * 65)
print('  v4.0.0 최종 결과')
print('=' * 65)
print(f'  [모델 구성]  CatBoost + LightGBM  (n_jobs={N_JOBS})')
print(f'  [검증 방법]  2022년 3월 holdout (테스트와 동일 계절)')
print(f'  [최종 선택]  {FINAL_MODE.upper()}')
print()
print('  [개별 모델 검증 MAE (2022년 3월)]')
print(f'    LightGBM  : {mae_lgb:>10,.2f} 원/kg  (best_iter={lgb_model.best_iteration_})')
print(f'    CatBoost  : {mae_cb:>10,.2f} 원/kg  (best_iter={cb_model.get_best_iteration()})')
print(f'    앙상블(avg): {mae_ens:>10,.2f} 원/kg')
print()
print('  [최종 검증 성능]')
print(f'    MAE  = {mae:>10,.2f} 원/kg')
print(f'    RMSE = {rmse:>10,.2f} 원/kg')
print(f'    R²   = {r2:>10.4f}')
print()
print(f'  제출 파일: {SUBMISSION_PATH}')
print('=' * 65)

### 다음 버전

| 버전 | 개선 내용 |
|------|----------|
| **v4.1.0** | Optuna로 LGB/CB 하이퍼파라미터 자동 최적화 (2022-03 검증 기준) |
| **v4.2.0** | 품목별 개별 모델 학습 (item-specific model) |
| **v5.0.0** | 그룹별 lag 피처 (전주/전달 가격) 추가 |